In [1]:
import importlib, time
simp = importlib.reload(simp)

# SIMP smoke test: 60x12 mesh, 1000 N tip load
t0 = time.time()
d = simp.optimize(60, 12, 0.4, 3.0, 1.5, "point", 1000.0, 0.0, maxiter=80)
ch = np.array(d['compliance'])
print(f"SIMP: {d['iterations']} iters in {time.time()-t0:.1f}s, "
      f"compliance {ch[0]:.1f} -> {ch[-1]:.1f} N-mm, vf={d['x'].mean():.3f}")

# Cross-check: full solid 500x100x1 mm beam under 1000 N tip load
# beam theory: I = 1*100^3/12, delta = PL^3/3EI, compliance = P*delta
I_sol = 100.0**3/12
d_sol = 1000*500**3/(3*200e3*I_sol)
print(f"solid unit-thickness beam: delta={d_sol:.2f} mm, compliance={1000*d_sol:.0f} N-mm")
print(f"SIMP iteration-1 (uniform 0.4) compliance={ch[0]:.1f} N-mm "
      f"(expect ~ {1000*d_sol/0.4**3:.0f} for uniform 0.4 density)")

# evaluate_design smoke check
rows = simp.evaluate_design(d, loads.LOAD_CASES)
r = rows["P1000"]
print(f"\nEval P1000: VM99={r['max_von_mises_MPa']:.1f} MPa, "
      f"delta={r['tip_deflection_mm']:.2f} mm, SF={r['safety_factor']:.3f}, "
      f"t={r['thickness_mm']:.2f} mm, I_eff={r['moment_of_inertia_mm4']:.0f} mm4")
import matplotlib.pyplot as plt
plt.figure(figsize=(8,2)); plt.imshow(-d['snapshots']['final'], cmap='gray'); plt.axis('off')
plt.savefig('/workspace/beamopt/smoke_topo.png', dpi=80, bbox_inches='tight'); plt.close()

SIMP: 80 iters in 0.4s, compliance 40058.7 -> 6559.2 N-mm, vf=0.399
solid unit-thickness beam: delta=2.50 mm, compliance=2500 N-mm
SIMP iteration-1 (uniform 0.4) compliance=40058.7 N-mm (expect ~ 39062 for uniform 0.4 density)

Eval P1000: VM99=140.0 MPa, delta=1.51 mm, SF=1.785, t=5.02 mm, I_eff=137744 mm4


In [2]:
import pandas as pd, pickle, time

# ---------- analytical sweep: 28 sections x 13 load cases ----------
reg = sections.build_geometry_registry()
rows = []
for gid, fam, lab, h, wfn, params in reg:
    props = sections.section_properties(h, wfn)
    for lc in loads.LOAD_CASES:
        r = loads.evaluate_section(props, lc)
        r.update(geometry_id=gid, geometry_family=fam, geometry_label=lab,
                 method="analytical")
        rows.append(r)
df_analytical = pd.DataFrame(rows)
print(f"analytical rows: {len(df_analytical)}")

# ---------- SIMP optimizations: 10 designs ----------
simp_runs = [
    ("SIMP-TIP-1", 100, 20, 1.5, "point",    1000.0, 0.0),
    ("SIMP-TIP-2", 150, 30, 1.5, "point",    1000.0, 0.0),
    ("SIMP-TIP-3", 100, 20, 3.0, "point",    1000.0, 0.0),
    ("SIMP-UDL-1", 100, 20, 1.5, "udl",      0.0, 50.0),
    ("SIMP-UDL-2", 150, 30, 1.5, "udl",      0.0, 50.0),
    ("SIMP-UDL-3", 100, 20, 3.0, "udl",      0.0, 50.0),
    ("SIMP-TRI-1", 100, 20, 1.5, "tri",      0.0, 50.0),
    ("SIMP-TRI-2", 150, 30, 1.5, "tri",      0.0, 50.0),
    ("SIMP-CMB-1", 100, 20, 1.5, "combined", 1000.0, 50.0),
    ("SIMP-CMB-2", 150, 30, 1.5, "combined", 1000.0, 50.0),
]
designs = {}
t0 = time.time()
for name, nx, ny, rmin, lt, P, w in simp_runs:
    t1 = time.time()
    d = simp.optimize(nx, ny, 0.4, 3.0, rmin, lt, P, w, maxiter=100)
    d["name"] = name
    designs[name] = d
    print(f"{name}: {nx}x{ny} {lt:8s} {d['iterations']:3d} iters, "
          f"C {d['compliance'][0]:.0f}->{d['compliance'][-1]:.0f}, "
          f"vf={d['x'].mean():.3f}  ({time.time()-t1:.0f}s)")
print(f"total SIMP time: {time.time()-t0:.0f}s")

# ---------- evaluate SIMP designs under all 13 load cases ----------
rows = []
for name, d in designs.items():
    case_rows = simp.evaluate_design(d, loads.LOAD_CASES)
    lt = d["opt_load"]["type"]
    lab = f"opt={lt},mesh={d['nelx']}x{d['nely']}"
    for cid, r in case_rows.items():
        r.update(geometry_id=name, geometry_family="SIMP freeform",
                 geometry_label=lab, method="simp-fea")
        rows.append(r)
df_simp = pd.DataFrame(rows)

# ---------- assemble, audit, save ----------
df = pd.concat([df_analytical, df_simp], ignore_index=True)
cols = ["geometry_id", "geometry_family", "geometry_label", "method",
        "load_case", "load_type", "point_load_N", "distributed_load_Nmm",
        "max_von_mises_MPa", "tip_deflection_mm", "safety_factor",
        "material_efficiency_index", "stress_uniformity_MPa", "mass_g",
        "moment_of_inertia_mm4", "area_mm2"]
df = df[cols]
assert len(df) == 494, len(df)
assert df.isna().sum().sum() == 0
assert (df.safety_factor > 0).all()
print(f"\ndataset: {df.shape}, NaN={int(df.isna().sum().sum())}, "
      f"families={df.geometry_family.nunique()}, "
      f"geometries={df.geometry_id.nunique()}, load cases={df.load_case.nunique()}")
df.to_csv("/workspace/beamopt/dataset.csv", index=False)
with open("/workspace/beamopt/designs.pkl", "wb") as f:
    pickle.dump({k: {kk: vv for kk, vv in v.items()} for k, v in designs.items()}, f)
print("saved /workspace/beamopt/dataset.csv and designs.pkl")

analytical rows: 364


SIMP-TIP-1: 100x20 point    100 iters, C 40185->4854, vf=0.400  (2s)


SIMP-TIP-2: 150x30 point    100 iters, C 40238->4364, vf=0.400  (4s)


SIMP-TIP-3: 100x20 point     63 iters, C 40185->7141, vf=0.398  (1s)


SIMP-UDL-1: 100x20 udl      100 iters, C 3912221->431730, vf=0.400  (1s)


...

SIMP-TRI-1: 100x20 tri      100 iters, C 330716->31913, vf=0.400  (1s)


SIMP-TRI-2: 150x30 tri      100 iters, C 331138->29743, vf=0.400  (4s)


SIMP-CMB-1: 100x20 combined 100 iters, C 4713310->535981, vf=0.400  (1s)


SIMP-CMB-2: 150x30 combined 100 iters, C 4717802->488331, vf=0.400  (4s)
total SIMP time: 24s



dataset: (494, 16), NaN=0, families=7, geometries=38, load cases=13
saved /workspace/beamopt/dataset.csv and designs.pkl


In [7]:
import pandas as pd, numpy as np
df = pd.read_csv('/workspace/beamopt/dataset.csv')
pd.set_option('display.width', 250)

# ---- 1) overall ranking by mean safety factor ----
rank = df.groupby(['geometry_id','geometry_family']).agg(
    mean_SF=('safety_factor','mean'), min_SF=('safety_factor','min'),
    mean_MEI=('material_efficiency_index','mean'),
    mean_defl=('tip_deflection_mm','mean'),
    mean_VM=('max_von_mises_MPa','mean'),
    I=('moment_of_inertia_mm4','first')).sort_values('mean_SF', ascending=False)
print('=== TOP 10 (mean safety factor across 13 load cases) ===')
print(rank.head(10).round(3).to_string())
print('\n=== BOTTOM 10 ===')
print(rank.tail(10).round(3).to_string())

# ---- 2) best geometry per load type ----
print('\n=== BEST PER LOAD TYPE (mean SF) ===')
for lt in ['point','udl','tri','combined']:
    sub = df[df.load_type==lt].groupby('geometry_id')['safety_factor'].mean().sort_values(ascending=False)
    best_an = next(i for i in sub.index if not i.startswith('SIMP'))
    print(f'{lt:9s}: overall {sub.index[0]:12s} SF={sub.iloc[0]:.3f} | best analytical: {best_an} SF={sub[best_an]:.3f}')

# ---- 3) load sensitivity ----
print('\n=== LOAD SENSITIVITY (point 100N -> 5000N) ===')
piv = df[df.load_type=='point'].pivot_table(index='geometry_id', columns='point_load_N', values='safety_factor')
piv['drop_pct'] = (piv[100.0]-piv[5000.0])/piv[100.0]*100
top5_ids = rank.head(5).index.get_level_values(0)
print(piv.loc[top5_ids].round(4).to_string())
print(f"linearity check: 100N->5000N SF drop = 98% for all rows: {np.allclose(piv['drop_pct'], 98.0)}")

# ---- 4) family summary ----
print('\n=== FAMILY x LOAD TYPE (mean SF) ===')
print(df.pivot_table(index='geometry_family', columns='load_type', values='safety_factor', aggfunc='mean').round(3).to_string())

# ---- 5) Pareto front on P1000 ----
sub = df[df.load_case=='P1000'][['geometry_id','tip_deflection_mm','max_von_mises_MPa']].reset_index(drop=True)
pareto = []
for i,r in sub.iterrows():
    dominated = ((sub.tip_deflection_mm<=r.tip_deflection_mm)&(sub.max_von_mises_MPa<=r.max_von_mises_MPa)&((sub.tip_deflection_mm<r.tip_deflection_mm)|(sub.max_von_mises_MPa<r.max_von_mises_MPa))).any()
    if not dominated: pareto.append(r.geometry_id)
print(f'\n=== PARETO-OPTIMAL (P1000): {pareto} ===')

# ---- 6) k-means k=4 ----
from sklearn.cluster import KMeans
feat = df.groupby('geometry_id').agg(
    mean_SF=('safety_factor','mean'), mean_defl=('tip_deflection_mm','mean'),
    mean_unif=('stress_uniformity_MPa','mean'), I=('moment_of_inertia_mm4','first'))
X = (feat - feat.mean())/feat.std()
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X)
feat['cluster'] = km.labels_
print('\n=== K-MEANS (k=4) ===')
for c in range(4):
    m = feat[feat.cluster==c]
    print(f"cluster {c}: n={len(m):2d}, mean_SF={m['mean_SF'].mean():.3f}, mean_defl={m['mean_defl'].mean():8.2f} mm, members={list(m.index)}")

# ---- 7) descriptive stats ----
print('\n=== DESCRIPTIVE STATISTICS ===')
print(df[['max_von_mises_MPa','tip_deflection_mm','safety_factor','material_efficiency_index','stress_uniformity_MPa','mass_g','moment_of_inertia_mm4']].describe().round(3).to_string())

# ---- 8) key scalars for the paper ----
best_overall = rank.index[0]; worst = rank.index[-1]
best_an_name = [i for i in rank.index if not i[0].startswith('SIMP')][0]
best_an_row = rank.loc[best_an_name]
print(f'\nbest overall: {best_overall}, best analytical: {best_an_name} (mean SF {best_an_row.mean_SF:.3f})')
print(f'SIMP improvement over best analytical: {(rank.loc[best_overall].mean_SF/best_an_row.mean_SF - 1)*100:.1f}%')
print(f'worst performer: {worst}')
rank.round(6).to_csv('/workspace/beamopt/ranking.csv')
feat.round(6).to_csv('/workspace/beamopt/clusters.csv')

=== TOP 10 (mean safety factor across 13 load cases) ===
                             mean_SF  min_SF  mean_MEI  mean_defl   mean_VM           I
geometry_id geometry_family                                                            
SIMP-TIP-2  SIMP freeform      3.457   0.073     0.017      6.361  1156.108  217437.047
SIMP-TIP-1  SIMP freeform      3.171   0.068     0.016      7.029  1236.061  197871.907
SIMP-TIP-3  SIMP freeform      2.035   0.051     0.010      9.566  1668.815  144488.553
SIMP-UDL-3  SIMP freeform      1.537   0.111     0.008     12.047   839.380   53577.084
SIMP-CMB-2  SIMP freeform      1.431   0.135     0.007      6.517   734.917  120168.835
SIMP-TRI-2  SIMP freeform      1.351   0.083     0.007     32.965  1049.115   10890.595
SIMP-CMB-1  SIMP freeform      1.296   0.139     0.006      7.143   756.988   97474.886
SIMP-UDL-1  SIMP freeform      1.172   0.115     0.006      8.477   835.045   67924.930
SIMP-UDL-2  SIMP freeform      1.149   0.114     0.006      7.7

In [6]:
import importlib, pickle, pandas as pd, numpy as np, time
simp = importlib.reload(simp)

# re-optimize all 10 designs WITH load-introduction skin
simp_runs = [
    ("SIMP-TIP-1", 100, 20, 1.5, "point",    1000.0, 0.0),
    ("SIMP-TIP-2", 150, 30, 1.5, "point",    1000.0, 0.0),
    ("SIMP-TIP-3", 100, 20, 3.0, "point",    1000.0, 0.0),
    ("SIMP-UDL-1", 100, 20, 1.5, "udl",      0.0, 50.0),
    ("SIMP-UDL-2", 150, 30, 1.5, "udl",      0.0, 50.0),
    ("SIMP-UDL-3", 100, 20, 3.0, "udl",      0.0, 50.0),
    ("SIMP-TRI-1", 100, 20, 1.5, "tri",      0.0, 50.0),
    ("SIMP-TRI-2", 150, 30, 1.5, "tri",      0.0, 50.0),
    ("SIMP-CMB-1", 100, 20, 1.5, "combined", 1000.0, 50.0),
    ("SIMP-CMB-2", 150, 30, 1.5, "combined", 1000.0, 50.0),
]
designs = {}
for name, nx, ny, rmin, lt, P, w in simp_runs:
    d = simp.optimize(nx, ny, 0.4, 3.0, rmin, lt, P, w, maxiter=100)
    d["name"] = name
    designs[name] = d
    print(f"{name}: {d['iterations']:3d} iters, C {d['compliance'][0]:.0f}->{d['compliance'][-1]:.0f}, vf={d['x'].mean():.3f}")

# evaluate under all 13 load cases
rows = []
for name, d in designs.items():
    case_rows = simp.evaluate_design(d, loads.LOAD_CASES)
    lab = f"opt={d['opt_load']['type']},mesh={d['nelx']}x{d['nely']}"
    for cid, r in case_rows.items():
        r.update(geometry_id=name, geometry_family="SIMP freeform",
                 geometry_label=lab, method="simp-fea")
        rows.append(r)
df_simp = pd.DataFrame(rows)
print('\nSIMP per-design means after fix:')
print(df_simp.groupby('geometry_id')[['tip_deflection_mm','max_von_mises_MPa','safety_factor','moment_of_inertia_mm4']].mean().round(3).to_string())

# rebuild full dataset
reg = sections.build_geometry_registry()
rows = []
for gid, fam, lab, h, wfn, params in reg:
    props = sections.section_properties(h, wfn)
    for lc in loads.LOAD_CASES:
        r = loads.evaluate_section(props, lc)
        r.update(geometry_id=gid, geometry_family=fam, geometry_label=lab, method="analytical")
        rows.append(r)
df = pd.concat([pd.DataFrame(rows), df_simp], ignore_index=True)
cols = ["geometry_id", "geometry_family", "geometry_label", "method",
        "load_case", "load_type", "point_load_N", "distributed_load_Nmm",
        "max_von_mises_MPa", "tip_deflection_mm", "safety_factor",
        "material_efficiency_index", "stress_uniformity_MPa", "mass_g",
        "moment_of_inertia_mm4", "area_mm2"]
df = df[cols]
assert len(df) == 494 and df.isna().sum().sum() == 0 and (df.safety_factor > 0).all()
assert df.tip_deflection_mm.max() < 1e4, df.tip_deflection_mm.max()
df.to_csv('/workspace/beamopt/dataset.csv', index=False)
with open('/workspace/beamopt/designs.pkl','wb') as f:
    pickle.dump(designs, f)
print(f"\ndataset rebuilt: {df.shape}, max deflection {df.tip_deflection_mm.max():.1f} mm")

SIMP-TIP-1: 100 iters, C 19254->4683, vf=0.403


SIMP-TIP-2: 100 iters, C 21419->4390, vf=0.401


SIMP-TIP-3: 100 iters, C 19254->5964, vf=0.406


SIMP-UDL-1: 100 iters, C 1930236->430465, vf=0.405


SIMP-UDL-2: 100 iters, C 2136068->401239, vf=0.402


...

SIMP-TRI-2: 100 iters, C 186387->30433, vf=0.403


SIMP-CMB-1: 100 iters, C 2316999->529018, vf=0.405


SIMP-CMB-2: 100 iters, C 2565792->487901, vf=0.402



SIMP per-design means after fix:
             tip_deflection_mm  max_von_mises_MPa  safety_factor  moment_of_inertia_mm4
geometry_id                                                                            
SIMP-CMB-1               7.143            756.988          1.296              97474.886
SIMP-CMB-2               6.517            734.917          1.431             120168.835
SIMP-TIP-1               7.029           1236.061          3.171             197871.907
SIMP-TIP-2               6.361           1156.108          3.457             217437.047
SIMP-TIP-3               9.566           1668.815          2.035             144488.553
SIMP-TRI-1              29.021           2205.110          0.800              14762.150
SIMP-TRI-2              32.965           1049.115          1.351              10890.595
SIMP-UDL-1               8.477            835.045          1.172              67924.930
SIMP-UDL-2               7.790            835.622          1.149              74686.80


dataset rebuilt: (494, 16), max deflection 3085.9 mm


In [5]:
import importlib
simp = importlib.reload(simp)
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import spsolve

# instrumented mini-run: replicate optimize() internals for 6 iterations
nelx, nely, volfrac, penal, rmin = 60, 12, 0.4, 3.0, 1.5
ndof = 2*(nelx+1)*(nely+1)
edof = np.zeros((nelx*nely, 8), dtype=int)
for elx in range(nelx):
    for ely in range(nely):
        e = elx*nely+ely
        n1 = (nely+1)*elx+ely; n2 = (nely+1)*(elx+1)+ely
        edof[e] = [2*n1,2*n1+1,2*n2,2*n2+1,2*n2+2,2*n2+3,2*n1+2,2*n1+3]
iK = np.kron(edof, np.ones((8,1),dtype=int)).ravel()
jK = np.kron(edof, np.ones((1,8),dtype=int)).ravel()
fixed = np.unique(np.concatenate([2*np.arange(nely+1), 2*np.arange(nely+1)+1]))
free = np.setdiff1d(np.arange(ndof), fixed)
F = simp.build_loads(nelx, nely, 'point', 1000.0, 0.0)
Hf, Hs = simp._filter(nelx, nely, rmin)
skin = simp.skin_mask(nelx, nely)
x = np.full(nelx*nely, volfrac); x[skin] = 1.0
KE = simp.KE; E0, EMIN = simp.E0, simp.EMIN

for loop in range(1, 7):
    Ee = EMIN + x**penal*(E0-EMIN)
    sK = (KE.ravel()[:,None]*Ee[None,:]).ravel(order='F')
    K = coo_matrix((sK,(iK,jK)), shape=(ndof,ndof)).tocsc()
    u = np.zeros(ndof); u[free] = spsolve(K[free][:,free], F[free])
    ce = ((u[edof].reshape(-1,8)@KE)*u[edof].reshape(-1,8)).sum(axis=1)
    dc = -penal*x**(penal-1)*(E0-EMIN)*ce
    dc = np.asarray(Hf.dot(x*dc)/Hs).ravel()/np.maximum(1e-3,x)
    print(f'it{loop}: u_nan={np.isnan(u).sum()}, ce_min={ce.min():.2e}, dc range=[{np.nanmin(dc):.2e},{np.nanmax(dc):.2e}], dc_nan={np.isnan(dc).sum()}')
    l1,l2,move = 0.0,1e9,0.2
    it_inner = 0
    while (l2-l1)/(l1+l2+1e-12) > 1e-3:
        lmid = 0.5*(l1+l2); it_inner += 1
        xnew = np.maximum(0.0, np.maximum(x-move, np.minimum(1.0, np.minimum(x+move, x*np.sqrt(-dc/lmid)))))
        xnew[skin] = 1.0
        if (xnew.mean()-volfrac) > 0: l1 = lmid
        else: l2 = lmid
        if it_inner > 100: print('  bisection stuck!'); break
    print(f'  bisection iters={it_inner}, lmid={lmid:.3e}, xnew mean={xnew.mean():.4f}, nan={np.isnan(xnew).sum()}')
    xphys = np.asarray(Hf.dot(xnew)/Hs).ravel(); xphys[skin]=1.0
    change = float(np.abs(xnew-x).max()); x = xnew
    print(f'  change={change:.4f}')

it1: u_nan=0, ce_min=4.43e-08, dc range=[-1.42e+03,-1.43e-01], dc_nan=0
  bisection iters=32, lmid=1.404e+02, xnew mean=0.3999, nan=0
  change=0.2000
it2: u_nan=0, ce_min=1.75e-07, dc range=[-6.08e+02,-1.31e+00], dc_nan=0
  bisection iters=32, lmid=1.488e+02, xnew mean=0.3999, nan=0
  change=0.2000
it3: u_nan=0, ce_min=4.46e-10, dc range=[-4.52e+02,-4.11e-01], dc_nan=0
  bisection iters=33, lmid=1.040e+02, xnew mean=0.4001, nan=0
  change=0.2000
it4: u_nan=0, ce_min=6.51e-12, dc range=[-3.35e+02,-9.70e-02], dc_nan=0
  bisection iters=33, lmid=9.302e+01, xnew mean=0.3999, nan=0
  change=0.2000
it5: u_nan=0, ce_min=1.35e-13, dc range=[-3.47e+02,-1.88e-02], dc_nan=0
  bisection iters=33, lmid=8.394e+01, xnew mean=0.4000, nan=0
  change=0.1557
it6: u_nan=0, ce_min=2.72e-15, dc range=[-3.11e+02,-2.12e-03], dc_nan=0
  bisection iters=33, lmid=8.254e+01, xnew mean=0.4000, nan=0
  change=0.1169


In [8]:
import json
df = pd.read_csv('/workspace/beamopt/dataset.csv')
rank = pd.read_csv('/workspace/beamopt/ranking.csv')
feat = pd.read_csv('/workspace/beamopt/clusters.csv')

print('=== BEST PER LOAD TYPE ===')
best_per_type = {}
for lt in ['point','udl','tri','combined']:
    sub = df[df.load_type==lt].groupby('geometry_id')['safety_factor'].mean().sort_values(ascending=False)
    best_an = next(i for i in sub.index if not i.startswith('SIMP'))
    best_per_type[lt] = dict(overall=sub.index[0], sf=float(sub.iloc[0]), best_analytical=best_an, sf_analytical=float(sub[best_an]))
    print(f'{lt:9s}: {sub.index[0]} SF={sub.iloc[0]:.3f} | analytical best: {best_an} SF={sub[best_an]:.3f}')

print('\n=== FAMILY x LOAD TYPE (mean SF) ===')
famtab = df.pivot_table(index='geometry_family', columns='load_type', values='safety_factor', aggfunc='mean')
print(famtab.round(3).to_string())

print('\n=== PARETO (P1000) ===')
sub = df[df.load_case=='P1000'][['geometry_id','tip_deflection_mm','max_von_mises_MPa']].reset_index(drop=True)
pareto = []
for i,r in sub.iterrows():
    dominated = ((sub.tip_deflection_mm<=r.tip_deflection_mm)&(sub.max_von_mises_MPa<=r.max_von_mises_MPa)&((sub.tip_deflection_mm<r.tip_deflection_mm)|(sub.max_von_mises_MPa<r.max_von_mises_MPa))).any()
    if not dominated: pareto.append(r.geometry_id)
print(pareto)

print('\n=== CLUSTERS ===')
clusters = {}
for c in range(4):
    m = feat[feat.cluster==c]
    clusters[c] = dict(n=len(m), mean_SF=float(m['mean_SF'].mean()), mean_defl=float(m['mean_defl'].mean()), members=list(m.geometry_id))
    print(c, clusters[c])

print('\n=== TOP5 / BOTTOM5 (mean SF) ===')
print(rank.head(5)[['geometry_id','mean_SF','min_SF','mean_defl','I']].to_string())
print(rank.tail(5)[['geometry_id','mean_SF','min_SF','mean_defl','I']].to_string())

# reference case table for paper (P1000, all geometries sorted by SF)
p1000 = df[df.load_case=='P1000'][['geometry_id','geometry_family','max_von_mises_MPa','tip_deflection_mm','safety_factor','moment_of_inertia_mm4']].sort_values('safety_factor', ascending=False)
print('\n=== P1000 reference: top 5 / bottom 5 ===')
print(p1000.head(5).round(3).to_string(index=False))
print(p1000.tail(5).round(3).to_string(index=False))

stats = dict(
    best_per_type=best_per_type, pareto=pareto, clusters=clusters,
    top10=rank.head(10).to_dict('records'), bottom10=rank.tail(10).to_dict('records'),
    family_table=famtab.round(4).to_dict(),
    desc=df[['max_von_mises_MPa','tip_deflection_mm','safety_factor','material_efficiency_index','stress_uniformity_MPa','moment_of_inertia_mm4']].describe().round(4).to_dict())
with open('/workspace/beamopt/stats.json','w') as f:
    json.dump(stats, f, indent=1, default=str)
print('\nsaved stats.json')

=== BEST PER LOAD TYPE ===
point    : SIMP-TIP-2 SF=10.318 | analytical best: CH-30 SF=3.078
udl      : SIMP-UDL-2 SF=0.634 | analytical best: IB-0.3 SF=0.147
tri      : SIMP-TRI-2 SF=2.192 | analytical best: IB-0.3 SF=0.435
combined : SIMP-CMB-1 SF=0.227 | analytical best: IB-0.3 SF=0.053

=== FAMILY x LOAD TYPE (mean SF) ===
load_type         combined  point    tri    udl
geometry_family                                
Circular hollow      0.025  1.788  0.155  0.069
Hollow rectangle     0.025  1.314  0.206  0.069
I-beam               0.050  2.631  0.411  0.138
SIMP freeform        0.153  4.145  1.416  0.444
Solid rectangle      0.014  0.710  0.111  0.037
T-section            0.021  1.152  0.156  0.057
Trapezoidal          0.013  0.689  0.108  0.036

=== PARETO (P1000) ===
['SIMP-TIP-2']

=== CLUSTERS ===
0 {'n': 12, 'mean_SF': 0.43452525000000003, 'mean_defl': 135.34156466666667, 'members': ['CH-12', 'CH-5', 'CH-8', 'HR-0.1', 'HR-0.2', 'HR-0.3', 'SR-0.5', 'TS-1', 'TS-2', 'TS-3', 'TS-

In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd, numpy as np, os
plt.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
plt.rcParams['svg.fonttype'] = 'none'
FIG = '/workspace/beamopt/figs'; os.makedirs(FIG, exist_ok=True)
df = pd.read_csv('/workspace/beamopt/dataset.csv')

FAM_COLORS = {'Solid rectangle':'#0072B2','I-beam':'#E69F00','Hollow rectangle':'#009E73',
              'T-section':'#CC79A7','Trapezoidal':'#D55E00','Circular hollow':'#56B4E9','SIMP freeform':'#000000'}

def save(fig, name):
    fig.savefig(f'{FIG}/{name}.svg', bbox_inches='tight')
    fig.savefig(f'{FIG}/{name}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

rank = df.groupby(['geometry_id','geometry_family']).agg(mean_SF=('safety_factor','mean')).reset_index().sort_values('mean_SF', ascending=False)

# ---- Graph 1: top-20 ranking bar chart ----
top20 = rank.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top20.geometry_id, top20.mean_SF, color=[FAM_COLORS[f] for f in top20.geometry_family], edgecolor='white')
ax.set_xlabel('Mean safety factor (13 load cases)')
ax.set_title('Top 20 geometries by material efficiency')
handles = [plt.Rectangle((0,0),1,1,color=c) for c in dict.fromkeys(top20.geometry_family.map(FAM_COLORS))]
labels = list(dict.fromkeys(top20.geometry_family))
ax.legend(handles, labels, loc='lower right', frameon=False, fontsize=9)
for i,(v,g) in enumerate(zip(top20.mean_SF, top20.geometry_id)):
    ax.text(v+0.02, i, f'{v:.2f}', va='center', fontsize=8)
ax.set_xlim(0, top20.mean_SF.max()*1.12)
save(fig, 'graph1_geometry_ranking')

# ---- Graph 2: Pareto front (P1000) ----
sub = df[df.load_case=='P1000']
fig, ax = plt.subplots(figsize=(8.5, 6))
for fam, g in sub.groupby('geometry_family'):
    ax.scatter(g.tip_deflection_mm, g.max_von_mises_MPa, s=55, color=FAM_COLORS[fam], label=fam, alpha=0.85, edgecolor='white', linewidth=0.5)
p = sub[sub.geometry_id=='SIMP-TIP-2'].iloc[0]
ax.annotate('SIMP-TIP-2\n(sole Pareto-optimal)', (p.tip_deflection_mm, p.max_von_mises_MPa),
            textcoords='offset points', xytext=(14,-30), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='0.3'))
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Tip deflection (mm)'); ax.set_ylabel('Max von Mises stress (MPa)')
ax.set_title('Deflection vs. stress, 1000 N tip load')
ax.legend(frameon=False, fontsize=9)
save(fig, 'graph2_pareto_front')

# ---- Graph 3: heatmap family x load case ----
order = ['P100','P500','P1000','P5000','UDL10','UDL50','UDL100','TRI10','TRI50','TRI100','C500+50','C1000+50','C1000+100']
h = df.pivot_table(index='geometry_family', columns='load_case', values='safety_factor', aggfunc='mean')[order]
fig, ax = plt.subplots(figsize=(11, 4.6))
im = ax.imshow(np.log10(h.values), aspect='auto', cmap='RdYlGn')
ax.set_xticks(range(len(order)), order, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(h.index)), h.index, fontsize=9)
cb = fig.colorbar(im, ax=ax, shrink=0.85)
ticks = cb.ax.get_yticks()
cb.ax.set_yticklabels([f'{10**t:.2g}' for t in ticks])
cb.set_label('Mean safety factor (log scale)')
for i in range(h.shape[0]):
    for j in range(h.shape[1]):
        ax.text(j, i, f'{h.values[i,j]:.2f}', ha='center', va='center', fontsize=7,
                color='black' if abs(np.log10(h.values[i,j]) - np.log10(h.values).mean()) < 0.7 else 'white')
ax.set_title('Safety factor by geometry family and load condition')
save(fig, 'graph3_heatmap_safety_factor')

# ---- Graph 4: SF vs point-load magnitude, top 5 ----
top5 = rank.head(5).geometry_id.tolist()
fig, ax = plt.subplots(figsize=(8.5, 5.5))
markers = ['o','s','^','D','v']
for g, mk in zip(top5, markers):
    s = df[(df.geometry_id==g)&(df.load_type=='point')].sort_values('point_load_N')
    fam = s.geometry_family.iloc[0]
    ax.plot(s.point_load_N, s.safety_factor, marker=mk, ms=5, lw=1.8, label=g, color=FAM_COLORS[fam] if fam!='SIMP freeform' else None)
ax.axhline(1.0, color='#D55E00', ls='--', lw=1.2)
ax.text(105, 1.06, 'SF = 1 (yield)', color='#D55E00', fontsize=9)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Tip point load (N)'); ax.set_ylabel('Safety factor')
ax.set_title('Safety factor vs. load magnitude, top 5 geometries')
ax.legend(frameon=False, fontsize=9)
save(fig, 'graph4_safety_vs_load')

# ---- Graph 6: boxplots by family ----
fig, ax = plt.subplots(figsize=(9.5, 5.5))
fams = ['SIMP freeform','I-beam','Circular hollow','Hollow rectangle','T-section','Solid rectangle','Trapezoidal']
data = [df[df.geometry_family==f].safety_factor.values for f in fams]
bp = ax.boxplot(data, labels=[f.replace(' ','\n') for f in fams], patch_artist=True, showfliers=False, medianprops=dict(color='black'))
for patch, f in zip(bp['boxes'], fams):
    patch.set_facecolor(FAM_COLORS[f]); patch.set_alpha(0.75)
ax.set_yscale('log')
ax.axhline(1.0, color='#D55E00', ls='--', lw=1.2)
ax.set_ylabel('Safety factor (log scale)')
ax.set_title('Safety factor distributions by geometry class (13 load cases each)')
save(fig, 'graph6_boxplots_geometry_class')
print('graphs 1,2,3,4,6 saved')
print(os.listdir(FIG))

/tmp/ipykernel_318/1708139068.py:58: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  cb.ax.set_yticklabels([f'{10**t:.2g}' for t in ticks])


/tmp/ipykernel_318/1708139068.py:87: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=[f.replace(' ','\n') for f in fams], patch_artist=True, showfliers=False, medianprops=dict(color='black'))


graphs 1,2,3,4,6 saved
['graph1_geometry_ranking.svg', 'graph1_geometry_ranking.png', 'graph2_pareto_front.svg', 'graph2_pareto_front.png', 'graph3_heatmap_safety_factor.svg', 'graph3_heatmap_safety_factor.png', 'graph4_safety_vs_load.svg', 'graph4_safety_vs_load.png', 'graph6_boxplots_geometry_class.svg', 'graph6_boxplots_geometry_class.png']


In [10]:
import pickle
with open('/workspace/beamopt/designs.pkl','rb') as f:
    designs = pickle.load(f)

# ---- Graph 5a-c: von Mises stress fields for top 3 geometries ----
top3 = ['SIMP-TIP-2', 'SIMP-TIP-1', 'SIMP-TIP-3']
suffix = ['a','b','c']
for name, sfx in zip(top3, suffix):
    d = designs[name]
    lt, P, w = d['opt_load']['type'], d['opt_load']['P'], d['opt_load']['w']
    u, F, _ = simp.solve_case(d, lt, P, w, eval_mode=False)
    vm = simp.element_vm(d, u)
    nelx, nely = d['nelx'], d['nely']
    ex, ey = 500.0/nelx, 100.0/nely
    t = 1e5 / (d['x'].sum()*ex*ey)
    vm = vm / t
    field = np.full(nelx*nely, np.nan)
    solid = d['x'] > 0.5
    field[solid] = vm[solid]
    field = field.reshape(nelx, nely).T
    vmax = np.nanpercentile(field, 99)
    fig, ax = plt.subplots(figsize=(11, 3.2))
    # void = light gray background
    ax.imshow(np.where(np.isnan(field), 1, np.nan), cmap='Greys', vmin=0, vmax=1,
              extent=[0,500,0,100], origin='upper', aspect='auto')
    im = ax.imshow(field, cmap='jet', vmin=0, vmax=vmax, extent=[0,500,0,100],
                   origin='upper', aspect='auto', interpolation='nearest')
    cb = fig.colorbar(im, ax=ax, shrink=0.9, pad=0.01)
    cb.set_label('von Mises stress (MPa)')
    load_desc = {'point': f'{P:.0f} N tip load', 'udl': f'{w:.0f} N/mm UDL',
                 'tri': f'{w:.0f} N/mm triangular', 'combined': f'{P:.0f} N + {w:.0f} N/mm'}[lt]
    ax.set_title(f'{name}: stress field under design load ({load_desc})')
    ax.set_xlabel('x (mm)'); ax.set_ylabel('y (mm)')
    ax.annotate('fixed end', (2, 50), fontsize=8, color='white', va='center')
    save(fig, f'graph5{sfx}_stress_map_top{suffix.index(sfx)+1}')
    print(f'graph5{sfx}: {name}, p99 VM = {vmax:.0f} MPa, t = {t:.2f} mm')

# ---- Graph 7: SIMP progression (SIMP-TIP-2) ----
d = designs['SIMP-TIP-2']
snaps = d['snapshots']
keys = [k for k in snaps if k != 'final']
fig, axes = plt.subplots(3, 3, figsize=(12, 5.2))
for ax, k in zip(axes.ravel(), keys):
    ax.imshow(-snaps[k], cmap='gray', aspect='auto')
    ax.set_title(f'iteration {k}', fontsize=10); ax.axis('off')
ax = axes.ravel()[len(keys)]
ax.imshow(-snaps['final'], cmap='gray', aspect='auto')
ax.set_title(f'final (iteration {d["iterations"]})', fontsize=10); ax.axis('off')
ax = axes.ravel()[len(keys)+1]
ax.plot(range(1, len(d['compliance'])+1), d['compliance'], color='#0066cc', lw=1.6)
ax.set_xlabel('iteration'); ax.set_ylabel('compliance (N-mm)')
ax.set_title('objective history', fontsize=10)
ax.set_yscale('log')
for ax in axes.ravel()[len(keys)+2:]:
    ax.axis('off')
fig.suptitle('SIMP topology optimization progression (SIMP-TIP-2, 150x30 mesh, p=3, vf=0.4)', fontsize=12)
fig.tight_layout()
save(fig, 'graph7_simp_progression')
print('graph7 saved, snapshots at iterations:', keys, '+ final')

graph5a: SIMP-TIP-2, p99 VM = 83 MPa, t = 4.98 mm
graph5b: SIMP-TIP-1, p99 VM = 92 MPa, t = 4.96 mm


graph5c: SIMP-TIP-3, p99 VM = 138 MPa, t = 4.92 mm


graph7 saved, snapshots at iterations: [1, 3, 5, 10, 20, 40] + final


In [11]:
# ---- Graph 3 (fixed): colorblind-safe cmap, luminance-aware text, 10^n ticks ----
order = ['P100','P500','P1000','P5000','UDL10','UDL50','UDL100','TRI10','TRI50','TRI100','C500+50','C1000+50','C1000+100']
h = df.pivot_table(index='geometry_family', columns='load_case', values='safety_factor', aggfunc='mean')[order]
lv = np.log10(h.values)
fig, ax = plt.subplots(figsize=(11, 4.6))
im = ax.imshow(lv, aspect='auto', cmap='cividis')
ax.set_xticks(range(len(order)), order, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(h.index)), h.index, fontsize=9)
vmin, vmax = lv.min(), lv.max()
cb = fig.colorbar(im, ax=ax, shrink=0.85)
tlo, thi = np.floor(vmin), np.ceil(vmax)
cb.set_ticks(np.arange(tlo, thi+1))
cb.ax.set_yticklabels([f'$10^{{{int(t)}}}$' for t in np.arange(tlo, thi+1)])
cb.set_label('Mean safety factor (log scale)')
for i in range(h.shape[0]):
    for j in range(h.shape[1]):
        frac = (lv[i,j]-vmin)/(vmax-vmin)
        ax.text(j, i, f'{h.values[i,j]:.2f}', ha='center', va='center', fontsize=7.5,
                color='white' if frac < 0.45 else 'black')
ax.set_title('Safety factor by geometry family and load condition')
save(fig, 'graph3_heatmap_safety_factor')

# ---- Graph 2 (fixed): move annotation clear of axes ----
sub = df[df.load_case=='P1000']
fig, ax = plt.subplots(figsize=(8.5, 6))
for fam, g in sub.groupby('geometry_family'):
    ax.scatter(g.tip_deflection_mm, g.max_von_mises_MPa, s=55, color=FAM_COLORS[fam], label=fam, alpha=0.85, edgecolor='white', linewidth=0.5)
p = sub[sub.geometry_id=='SIMP-TIP-2'].iloc[0]
ax.annotate('SIMP-TIP-2\n(sole Pareto-optimal)', (p.tip_deflection_mm, p.max_von_mises_MPa),
            textcoords='offset points', xytext=(18, 26), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='0.3'))
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Tip deflection (mm)'); ax.set_ylabel('Max von Mises stress (MPa)')
ax.set_title('Deflection vs. stress, 1000 N tip load')
ax.legend(frameon=False, fontsize=9, loc='upper right')
save(fig, 'graph2_pareto_front')
print('graphs 2,3 regenerated')

graphs 2,3 regenerated


In [12]:
# ---- Graph 2 (final): legend outside plot area ----
sub = df[df.load_case=='P1000']
fig, ax = plt.subplots(figsize=(9.2, 6))
for fam, g in sub.groupby('geometry_family'):
    ax.scatter(g.tip_deflection_mm, g.max_von_mises_MPa, s=55, color=FAM_COLORS[fam], label=fam, alpha=0.85, edgecolor='white', linewidth=0.5)
p = sub[sub.geometry_id=='SIMP-TIP-2'].iloc[0]
ax.annotate('SIMP-TIP-2\n(sole Pareto-optimal)', (p.tip_deflection_mm, p.max_von_mises_MPa),
            textcoords='offset points', xytext=(16, 22), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='0.3'))
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Tip deflection (mm)'); ax.set_ylabel('Max von Mises stress (MPa)')
ax.set_title('Deflection vs. stress, 1000 N tip load')
ax.legend(frameon=False, fontsize=9, loc='center left', bbox_to_anchor=(1.01, 0.5))
save(fig, 'graph2_pareto_front')

# ---- Graph 3 (final): add axis titles ----
fig, ax = plt.subplots(figsize=(11, 4.6))
im = ax.imshow(lv, aspect='auto', cmap='cividis')
ax.set_xticks(range(len(order)), order, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(h.index)), h.index, fontsize=9)
cb = fig.colorbar(im, ax=ax, shrink=0.85)
cb.set_ticks(np.arange(tlo, thi+1))
cb.ax.set_yticklabels([f'$10^{{{int(t)}}}$' for t in np.arange(tlo, thi+1)])
cb.set_label('Mean safety factor (log scale)')
for i in range(h.shape[0]):
    for j in range(h.shape[1]):
        frac = (lv[i,j]-vmin)/(vmax-vmin)
        ax.text(j, i, f'{h.values[i,j]:.2f}', ha='center', va='center', fontsize=7.5,
                color='white' if frac < 0.5 else 'black')
ax.set_title('Safety factor by geometry family and load condition')
ax.set_xlabel('Load condition'); ax.set_ylabel('Geometry family')
save(fig, 'graph3_heatmap_safety_factor')

# ---- Graph 5a-c (final): turbo colormap, explicit white voids ----
for name, sfx in zip(top3, ['a','b','c']):
    d = designs[name]
    lt, P, w = d['opt_load']['type'], d['opt_load']['P'], d['opt_load']['w']
    u, F, _ = simp.solve_case(d, lt, P, w, eval_mode=False)
    vm = simp.element_vm(d, u)
    nelx, nely = d['nelx'], d['nely']
    ex, ey = 500.0/nelx, 100.0/nely
    t = 1e5 / (d['x'].sum()*ex*ey)
    vm = vm / t
    field = np.full(nelx*nely, np.nan)
    field[d['x'] > 0.5] = vm[d['x'] > 0.5]
    field = field.reshape(nelx, nely).T
    vmax = np.nanpercentile(field, 99)
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.set_facecolor('white')
    cmap = plt.get_cmap('turbo').copy()
    cmap.set_bad(color='white')
    im = ax.imshow(np.ma.masked_invalid(field), cmap=cmap, vmin=0, vmax=vmax,
                   extent=[0,500,0,100], origin='upper', aspect='auto', interpolation='nearest')
    cb = fig.colorbar(im, ax=ax, shrink=0.9, pad=0.01)
    cb.set_label('von Mises stress (MPa)')
    load_desc = {'point': f'{P:.0f} N tip load', 'udl': f'{w:.0f} N/mm UDL',
                 'tri': f'{w:.0f} N/mm triangular', 'combined': f'{P:.0f} N + {w:.0f} N/mm'}[lt]
    ax.set_title(f'{name}: stress field under design load ({load_desc})')
    ax.set_xlabel('x (mm)'); ax.set_ylabel('y (mm)')
    ax.annotate('fixed end', (6, 8), fontsize=8, color='0.25')
    save(fig, f'graph5{sfx}_stress_map_top{["a","b","c"].index(sfx)+1}')
print('graphs 2, 3, 5a-c finalized')

graphs 2, 3, 5a-c finalized


In [13]:
p1000 = df[df.load_case=='P1000'][['geometry_id','geometry_family','max_von_mises_MPa','tip_deflection_mm','safety_factor','moment_of_inertia_mm4']].sort_values('safety_factor', ascending=False)
print('P1000 top 10:')
print(p1000.head(10).round(3).to_string(index=False))
print('\nUDL50 top 5:')
print(df[df.load_case=='UDL50'][['geometry_id','max_von_mises_MPa','tip_deflection_mm','safety_factor']].sort_values('safety_factor',ascending=False).head(5).round(3).to_string(index=False))
print('\nTRI50 top 5:')
print(df[df.load_case=='TRI50'][['geometry_id','max_von_mises_MPa','tip_deflection_mm','safety_factor']].sort_values('safety_factor',ascending=False).head(5).round(3).to_string(index=False))
print('\nC1000+50 top 5:')
print(df[df.load_case=='C1000+50'][['geometry_id','max_von_mises_MPa','tip_deflection_mm','safety_factor']].sort_values('safety_factor',ascending=False).head(5).round(3).to_string(index=False))
# I values for key analytical sections
for g in ['SR-1.0','IB-0.3','IB-1.0','HR-0.1','CH-30','CH-5','TS-1','TR-0.5']:
    print(g, 'I =', round(df[df.geometry_id==g].moment_of_inertia_mm4.iloc[0],1), 'mm4')
# SIMP vs analytical means by load type (family table already have); SIMP-TIP-2 deflection P1000
print('\nSIMP-TIP-2 P1000:', p1000[p1000.geometry_id=='SIMP-TIP-2'].round(4).to_string(index=False))
# fraction of rows with SF<1
print('\nrows with SF<1:', (df.safety_factor<1).sum(), '/', len(df))
# uniformity: most uniform under P1000
print('\nmost uniform (P1000, lowest stress std):')
print(df[df.load_case=='P1000'][['geometry_id','stress_uniformity_MPa']].sort_values('stress_uniformity_MPa').head(5).round(2).to_string(index=False))

P1000 top 10:
geometry_id geometry_family  max_von_mises_MPa  tip_deflection_mm  safety_factor  moment_of_inertia_mm4
 SIMP-TIP-2   SIMP freeform             79.954              0.958          3.127             217437.047
 SIMP-TIP-1   SIMP freeform             87.442              1.053          2.859             197871.907
 SIMP-TIP-3   SIMP freeform            137.783              1.442          1.814             144488.553
 SIMP-UDL-3   SIMP freeform            234.450              3.888          1.066              53577.084
      CH-30 Circular hollow            267.988              4.509          0.933              46206.667
     IB-0.3          I-beam            293.887              6.912          0.851              30139.781
 SIMP-CMB-2   SIMP freeform            298.678              1.734          0.837             120168.835
     IB-0.5          I-beam            301.874              8.335          0.828              24996.352
     IB-0.7          I-beam            318.064    